# AUTokens50 Chonkie Chunking Exploration

This notebook focuses on the 2026-08-31 direction: pause SimHash/MinHash reduction work and study chunking with the open source Chonkie library.

The workflow is intentionally a bounded preview. It reads only a small sample from `AUTokens50_with_hash_simhash/part_*.parquet`, keeps source data read-only, compares Chonkie chunkers, and writes reviewable output files outside the source directory.

If the AUTokens50 parquet directory is not available, the notebook falls back to a few sample texts so the chunking process remains inspectable.

## Stakeholder Requirements

This notebook is anchored to the stakeholder request for this week:

- Last week's duplicate analysis was accepted as accurate; around 70% duplication was independently confirmed.
- Do not continue reducing the dataset with SimHash for this workstream.
- Focus on understanding the chunking process.
- Use the open source Chonkie library as the chunking tool.
- Run or continue the notebook on the stakeholder-provided JupyterHub server: https://jupyterhub.southerncross.ai
- If the available environment and data allow it, produce chunked sample outputs.
- If full chunking cannot be completed, leave the notebooks visible and runnable so the next person can continue.

## Target JupyterHub Environment

Stakeholder-provided server: https://jupyterhub.southerncross.ai

Expected workflow there:

1. Open this notebook on JupyterHub.
2. Run Cell 1 to install any missing packages into the active kernel.
3. Run Cell 3 to confirm the AUTokens50 parquet path is discovered.
4. Keep the default sample limits for the first run.
5. Review the saved preview files under `Scratch/AUTokens50_chonkie_chunk_preview`.
6. Increase sample limits only after confirming chunk quality and available storage.

## References

- Classmate reference notebook: https://github.com/joeyllm/JoeyLLM-Team/blob/main/team-members/xingyu-li/notebooks/AUTokens50_chonkie_chunking_exploration.ipynb
- Chonkie GitHub repository: https://github.com/feyninc/chonkie
- Chonkie installation notes: https://docs.chonkie.ai/oss/installation
- Chonkie chunker overview: https://docs.chonkie.ai/oss/chunkers/choosing-a-chunker

## Cell 1 - Dependency Check

The online Jupyter environment may already include `pandas` and `pyarrow`. This cell installs only missing packages into the active notebook kernel.

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

required_packages = {
    "chonkie": "chonkie==1.7.0",
    "pandas": "pandas>=2.2",
    "pyarrow": "pyarrow>=17",
}

missing = [pip_name for import_name, pip_name in required_packages.items() if importlib.util.find_spec(import_name) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are already available.")

## Cell 2 - Imports And Configuration

The first pass uses a character tokenizer so the preview can run without downloading tokenizer model assets. Increase limits only after reviewing the preview quality.

In [ ]:
from pathlib import Path
from dataclasses import dataclass
import hashlib
import json
import math
import re
import time
from typing import Iterable

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from chonkie import FastChunker, RecursiveChunker, SentenceChunker, TokenChunker

TEXT_COL = "text"
HASH_COL = "hash"
SIMHASH_COL = "simhash"

MAX_FILES = 2
MAX_ROWS_PER_FILE = 25
MAX_DOCS_TO_CHUNK = 20
PREVIEW_TEXT_CHARS = 220

CHUNK_SIZE = 2048
CHUNK_OVERLAP = 128
TOKENIZER = "character"

INPUT_DIR_OVERRIDE = None
OUTPUT_DIR_OVERRIDE = None

input_candidates = [
    Path.cwd() / "JoeyLLM_Data/AUTokens50_with_hash_simhash",
    Path("/JoeyLLM_Data/AUTokens50_with_hash_simhash"),
    Path("/home/jovyan/JoeyLLM_Data/AUTokens50_with_hash_simhash"),
    Path("/home/jovyan/notebook/JoeyLLM_Data/AUTokens50_with_hash_simhash"),
    Path("/home/jovyan/data/JoeyLLM_Data/AUTokens50_with_hash_simhash"),
    Path("/home/jovyan/JoeyLLM_data/AUTokens50_with_hash_simhash"),
    Path("/home/jovyan/notebook/JoeyLLM_data/AUTokens50_with_hash_simhash"),
    Path("/home/jovyan/data/JoeyLLM_data/AUTokens50_with_hash_simhash"),
]

scratch_candidates = [
    Path.cwd() / "Scratch",
    Path("/Scratch"),
    Path("/home/jovyan/Scratch"),
    Path("/home/jovyan/notebook/Scratch"),
    Path.cwd() / "chunking_output",
]

fallback_texts = [
    "JoeyLLM needs reliable chunking before embeddings and vector database loading. This preview uses Chonkie locally and records chunk metadata for review.",
    "The 70 percent duplication finding was validated separately, so this notebook does not reduce the corpus. It focuses only on chunking behavior.",
    "A useful retrieval chunk should preserve enough context to answer a question while staying small enough for embedding and retrieval efficiency.",
]

print({
    "max_files": MAX_FILES,
    "max_rows_per_file": MAX_ROWS_PER_FILE,
    "max_docs_to_chunk": MAX_DOCS_TO_CHUNK,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "tokenizer": TOKENIZER,
})

## Cell 3 - Resolve Dataset And Output Paths

This cell discovers parquet part files and prepares an output folder. It does not modify the source dataset.

In [ ]:
def part_sort_key(path: Path) -> tuple[int, str]:
    try:
        return int(path.stem.rsplit("_", 1)[-1]), path.name
    except ValueError:
        return 10**12, path.name


def first_existing(paths: Iterable[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


input_dir = Path(INPUT_DIR_OVERRIDE).expanduser().resolve() if INPUT_DIR_OVERRIDE else first_existing(input_candidates)
scratch_base = Path(OUTPUT_DIR_OVERRIDE).expanduser().resolve() if OUTPUT_DIR_OVERRIDE else first_existing(scratch_candidates)
if scratch_base is None:
    scratch_base = Path.cwd() / "chunking_output"
output_dir = scratch_base / "AUTokens50_chonkie_chunk_preview"
output_dir.mkdir(parents=True, exist_ok=True)

if input_dir is None:
    parquet_files = []
    print("No AUTokens50 parquet directory found. Fallback sample text will be used.")
else:
    parquet_files = sorted(input_dir.glob("part_*.parquet"), key=part_sort_key)
    print("Input directory:", input_dir)
    print("Parquet part files found:", len(parquet_files))

print("Output directory:", output_dir)
print("First files:", [path.name for path in parquet_files[:5]])

## Cell 4 - Load A Bounded Sample

Large `part_*.parquet` files should not be loaded wholesale for this preview. The helper below reads only selected columns and only the first small batch from each sampled file.

In [ ]:
def read_rows_from_part(path: Path, max_rows: int) -> pd.DataFrame:
    parquet_file = pq.ParquetFile(path)
    schema_names = parquet_file.schema_arrow.names
    if TEXT_COL not in schema_names:
        raise ValueError(f"{path} is missing required text column: {TEXT_COL}")

    columns = [col for col in [TEXT_COL, HASH_COL, SIMHASH_COL] if col in schema_names]
    rows_left = max_rows if max_rows > 0 else None
    tables = []

    batch_size = min(max_rows, 1024) if max_rows > 0 else 1024
    for batch in parquet_file.iter_batches(batch_size=batch_size, columns=columns):
        table = pa.Table.from_batches([batch])
        if rows_left is not None and table.num_rows > rows_left:
            table = table.slice(0, rows_left)
        tables.append(table)
        if rows_left is not None:
            rows_left -= table.num_rows
        if rows_left is not None and rows_left <= 0:
            break

    if not tables:
        return pd.DataFrame(columns=["source_file", *columns])

    frame = pa.concat_tables(tables).to_pandas()
    frame.insert(0, "source_file", path.name)
    return frame


sample_frames = []
for part_path in parquet_files[:MAX_FILES]:
    sample_frames.append(read_rows_from_part(part_path, MAX_ROWS_PER_FILE))

if sample_frames:
    docs_df = pd.concat(sample_frames, ignore_index=True)
    docs_df = docs_df[docs_df[TEXT_COL].notna()].copy().head(MAX_DOCS_TO_CHUNK)
    docs_df[TEXT_COL] = docs_df[TEXT_COL].astype(str)
    data_mode = "parquet_sample"
else:
    docs_df = pd.DataFrame({
        "source_file": ["fallback_sample"] * len(fallback_texts),
        TEXT_COL: fallback_texts,
        HASH_COL: [None] * len(fallback_texts),
        SIMHASH_COL: [None] * len(fallback_texts),
    })
    data_mode = "fallback_sample"

print("Data mode:", data_mode)
print("Documents loaded:", len(docs_df))
docs_df[["source_file", TEXT_COL]].assign(text_preview=lambda df: df[TEXT_COL].str.slice(0, PREVIEW_TEXT_CHARS))[["source_file", "text_preview"]].head(5)

## Cell 5 - Compare Baseline And Extra Chunking Strategies

The first three strategies are the direct Chonkie baselines. The extra strategies cut the text in different ways so this notebook can explore more than the previous version:

- `token`: fixed-size Chonkie token or character windows.
- `sentence`: Chonkie sentence-aware grouping.
- `recursive`: Chonkie hierarchical splitting.
- `fast`: Chonkie's high-throughput byte-oriented chunking baseline.
- `paragraph_recursive`: merge natural paragraphs first, then recursively split oversized blocks.
- `boundary_recursive`: detect likely article/post boundaries first, then recursively split each section.

In [ ]:
@dataclass
class NotebookChunk:
    text: str
    token_count: int | None
    start_index: int | None
    end_index: int | None


def count_for_active_tokenizer(text: str) -> int | None:
    if TOKENIZER == "character":
        return len(text)
    return None


def paragraph_spans(text: str) -> list[tuple[int, int]]:
    spans = []
    for match in re.finditer(r"[^\S\n]*\S.*?(?=\n\s*\n|\Z)", text, flags=re.DOTALL):
        raw = match.group(0)
        start = match.start() + len(raw) - len(raw.lstrip())
        end = match.start() + len(raw.rstrip())
        if start < end:
            spans.append((start, end))
    return spans or [(0, len(text))]


def pack_spans(text: str, spans: list[tuple[int, int]], max_chars: int) -> list[tuple[int, int, str]]:
    packed = []
    current_start = None
    current_end = None
    for start, end in spans:
        if current_start is None:
            current_start, current_end = start, end
            continue
        if end - current_start > max_chars and current_end is not None:
            packed.append((current_start, current_end, text[current_start:current_end]))
            current_start, current_end = start, end
        else:
            current_end = end
    if current_start is not None and current_end is not None:
        packed.append((current_start, current_end, text[current_start:current_end]))
    return packed


class ParagraphRecursiveChunker:
    def __init__(self, tokenizer: str, chunk_size: int, chunk_overlap: int):
        self.chunk_size = chunk_size
        self.inner = RecursiveChunker(tokenizer=tokenizer, chunk_size=chunk_size, chunk_overlap=chunk_overlap)

    def split_spans(self, text: str) -> list[tuple[int, int]]:
        return paragraph_spans(text)

    def __call__(self, text: str) -> list[NotebookChunk]:
        chunks = []
        for start, _, block in pack_spans(text, self.split_spans(text), self.chunk_size):
            block_text = block.strip()
            if not block_text:
                continue
            block_start = start + len(block) - len(block.lstrip())
            if len(block_text) <= self.chunk_size:
                chunks.append(NotebookChunk(block_text, count_for_active_tokenizer(block_text), block_start, block_start + len(block_text)))
                continue
            for inner_chunk in self.inner(block_text):
                chunk_text = str(inner_chunk.text).strip()
                if not chunk_text:
                    continue
                local_start = getattr(inner_chunk, "start_index", None)
                local_end = getattr(inner_chunk, "end_index", None)
                chunks.append(NotebookChunk(
                    chunk_text,
                    getattr(inner_chunk, "token_count", count_for_active_tokenizer(chunk_text)),
                    block_start + local_start if isinstance(local_start, int) else None,
                    block_start + local_end if isinstance(local_end, int) else None,
                ))
        return chunks


class BoundaryRecursiveChunker(ParagraphRecursiveChunker):
    boundary_pattern = re.compile(
        r"(?m)(?=^(?:#{1,6}\s+.+|.+(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),\s+[A-Z][a-z]+,\s+\d{1,2},\s+\d{4}.*)$)"
    )

    def split_spans(self, text: str) -> list[tuple[int, int]]:
        starts = sorted({0, *(match.start() for match in self.boundary_pattern.finditer(text))})
        spans = []
        for start, end in zip(starts, [*starts[1:], len(text)]):
            segment = text[start:end]
            clean_start = start + len(segment) - len(segment.lstrip())
            clean_end = start + len(segment.rstrip())
            if clean_start < clean_end:
                spans.append((clean_start, clean_end))
        return spans if len(spans) > 1 else paragraph_spans(text)


chunkers = {
    "token": TokenChunker(tokenizer=TOKENIZER, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP),
    "sentence": SentenceChunker(tokenizer=TOKENIZER, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP),
    "recursive": RecursiveChunker(tokenizer=TOKENIZER, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP),
    "fast": FastChunker(chunk_size=CHUNK_SIZE),
    "paragraph_recursive": ParagraphRecursiveChunker(TOKENIZER, CHUNK_SIZE, CHUNK_OVERLAP),
    "boundary_recursive": BoundaryRecursiveChunker(TOKENIZER, CHUNK_SIZE, CHUNK_OVERLAP),
}

ENABLE_SEMANTIC_EXPERIMENT = False
if ENABLE_SEMANTIC_EXPERIMENT:
    from chonkie import SemanticChunker

    chunkers["semantic_optional"] = SemanticChunker(chunk_size=CHUNK_SIZE)

chunkers

## Cell 6 - Chunk The Sample And Capture Metadata

Each chunk keeps the original document index, source file, optional hash fields, chunk position, approximate offsets from Chonkie, and a short preview.

In [ ]:
def digest(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def value_or_none(row: pd.Series, column: str) -> object:
    if column not in row.index:
        return None
    value = row[column]
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    return value


records = []
started = time.time()
for doc_index, row in docs_df.reset_index(drop=True).iterrows():
    text = str(row[TEXT_COL]).strip()
    if not text:
        continue

    source_file = str(row.get("source_file", "unknown"))
    source_hash = value_or_none(row, HASH_COL)
    source_simhash = value_or_none(row, SIMHASH_COL)
    source_text_hash = digest(text)

    for chunker_name, chunker in chunkers.items():
        for chunk_index, chunk in enumerate(chunker(text)):
            chunk_text = str(chunk.text).strip()
            if not chunk_text:
                continue
            records.append({
                "chunk_id": digest(f"{source_file}:{doc_index}:{chunker_name}:{chunk_index}:{chunk_text}")[:20],
                "chunker": chunker_name,
                "doc_index": int(doc_index),
                "source_file": source_file,
                "source_hash": source_hash,
                "source_simhash": source_simhash,
                "source_text_sha256": source_text_hash,
                "chunk_index": int(chunk_index),
                "chunk_token_count": getattr(chunk, "token_count", None),
                "chunk_char_count": len(chunk_text),
                "start_index": getattr(chunk, "start_index", None),
                "end_index": getattr(chunk, "end_index", None),
                "chunk_preview": chunk_text[:PREVIEW_TEXT_CHARS].replace("\n", " "),
                "chunk_text": chunk_text,
            })

chunks_df = pd.DataFrame(records)
elapsed_seconds = round(time.time() - started, 3)
print("Preview chunks:", len(chunks_df))
print("Elapsed seconds:", elapsed_seconds)
chunks_df.head()

## Cell 7 - Summarize Chunk Shape

The first review should look for chunk count, average size, minimum-size tails, and whether chunks split awkwardly.

In [ ]:
def mean_or_none(series: pd.Series) -> float | None:
    cleaned = series.dropna()
    if cleaned.empty:
        return None
    return round(float(cleaned.mean()), 2)


summary_df = (
    chunks_df.groupby("chunker")
    .agg(
        documents=("doc_index", "nunique"),
        chunks=("chunk_index", "count"),
        avg_chunks_per_doc=("doc_index", lambda s: round(len(s) / max(1, s.nunique()), 2)),
        min_chunk_chars=("chunk_char_count", "min"),
        avg_chunk_chars=("chunk_char_count", lambda s: round(float(s.mean()), 2)),
        max_chunk_chars=("chunk_char_count", "max"),
        avg_token_count=("chunk_token_count", mean_or_none),
    )
    .reset_index()
    .sort_values("chunker")
)

summary_df

## Cell 8 - Save Review Outputs

The parquet file keeps full chunk text for later inspection. The CSV omits full text and is meant as a lightweight audit index.

In [ ]:
preview_parquet = output_dir / "chonkie_chunk_preview.parquet"
preview_csv = output_dir / "chonkie_chunk_preview_index.csv"
summary_json = output_dir / "chonkie_chunk_preview_summary.json"

chunks_df.to_parquet(preview_parquet, index=False)
chunks_df.drop(columns=["chunk_text"], errors="ignore").to_csv(preview_csv, index=False)

summary = {
    "task": "AUTokens50 Chonkie chunking preview",
    "email_date": "2026-08-31",
    "data_mode": data_mode,
    "input_dir": str(input_dir) if input_dir else None,
    "output_dir": str(output_dir),
    "files_seen": len(parquet_files),
    "files_sampled": min(MAX_FILES, len(parquet_files)),
    "documents_sampled": int(len(docs_df)),
    "chunkers": list(chunkers.keys()),
    "tokenizer": TOKENIZER,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "preview_chunks": int(len(chunks_df)),
    "elapsed_seconds": elapsed_seconds,
    "metrics": summary_df.to_dict(orient="records"),
    "outputs": {
        "preview_parquet": str(preview_parquet),
        "preview_csv": str(preview_csv),
        "summary_json": str(summary_json),
    },
    "notes": [
        "Source parquet files were treated as read-only.",
        "This is a bounded preview for understanding chunking, not a full production run.",
        "Hash and simhash fields are retained as metadata only; no duplicate reduction is performed here.",
    ],
}

summary_json.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Cell 9 - Manual Review Samples

Use this output to decide whether chunk boundaries are useful before scaling to more files or all rows.

In [ ]:
for chunker_name in chunkers:
    print("\n" + "=" * 80)
    print("Chunker:", chunker_name)
    examples = chunks_df[chunks_df["chunker"] == chunker_name].head(3)
    for _, item in examples.iterrows():
        print("-" * 80)
        print({
            "doc_index": int(item["doc_index"]),
            "chunk_index": int(item["chunk_index"]),
            "chars": int(item["chunk_char_count"]),
            "tokens": None if pd.isna(item["chunk_token_count"]) else int(item["chunk_token_count"]),
        })
        print(item["chunk_preview"])

## Decision Notes

Initial recommendation for the next review:

- Keep this as a preview until the approved source dataset or deduplicated dataset is confirmed.
- Keep `TokenChunker`, `SentenceChunker`, and `RecursiveChunker` as direct Chonkie baselines.
- Add `fast` to understand the speed-oriented fixed-window behavior.
- Add `paragraph_recursive` to test whether natural paragraph boundaries produce cleaner retrieval chunks.
- Add `boundary_recursive` to test whether likely article or post boundaries improve AUTokens50 chunks, especially when one row contains multiple concatenated posts.
- Keep the optional `semantic_optional` experiment disabled by default because it may require extra dependencies or model downloads on JupyterHub.
- Do not claim the full corpus has been chunked until a separate production run processes all approved parquet parts and validates output counts.

Scaling step once approved:

```python
# Increase MAX_FILES and MAX_ROWS_PER_FILE, or remove sampling only after checking storage and runtime.
MAX_FILES = 57
MAX_ROWS_PER_FILE = 0  # only if the production loop is changed to stream/write incrementally
```

For a production run, prefer a script that streams file-by-file and writes chunks incrementally instead of keeping all chunks in memory.